# Upscale — data preprocessing

Turns source frames into the `(low resolution, high resolution)` patch pairs the
upscaling model trains on, and writes them next to a manifest that
`upscale_dummy.ipynb` reads back.

The pipeline is four steps:

1. **Collect** every image under `data/raw/` — screenshots, or frames pulled out of a
   recording with ffmpeg. With nothing there it synthesises desktop-like frames instead,
   so the notebook runs end to end before a dataset exists.
2. **Degrade** each frame: crop it to a multiple of `SCALE`, that crop is the HR target,
   and a bicubic downscale of it is the LR input.
3. **Tile** both into aligned patches, dropping the near-flat ones — a desktop frame is
   mostly empty background, and a model trained on it learns to copy flat colour.
4. **Save** four `uint8` arrays (train/val × lr/hr) plus `manifest.json`.

The split is by *source frame*, not by patch: neighbouring patches of one frame overlap,
so splitting after tiling would put near-copies of the training data into validation and
every metric downstream would read high.

In [ ]:
"""Configuration — every knob the notebook has."""

from pathlib import Path

# Paths are relative to the folder of this notebook (model/upscale).
RAW_DIR = Path("data/raw")
OUT_DIR = Path("data/processed")

SCALE = 2                      # upscaling factor the model will be trained for
PATCH_HR = 128                 # HR patch edge in pixels; LR patch is PATCH_HR // SCALE
STRIDE_HR = 96                 # tiling stride in HR pixels (below PATCH_HR they overlap)
VAR_MIN = 24.0                 # drop a patch whose luma variance is below this
MAX_PATCHES_PER_FRAME = 64     # cap, so one large frame cannot dominate the set
VAL_FRACTION = 0.2             # share of *source frames* held out for validation
SEED = 0

# With data/raw empty, synthesise this many frames rather than producing nothing.
SYNTHESISE_IF_EMPTY = True
SYNTHETIC_FRAMES = 12
SYNTHETIC_SIZE = (960, 600)

assert PATCH_HR % SCALE == 0, "PATCH_HR must be a multiple of SCALE"
PATCH_LR = PATCH_HR // SCALE

In [ ]:
"""Imports, folders, and the RNG everything below draws from."""

import json
import time

import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

rng = np.random.default_rng(SEED)

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("raw       ", RAW_DIR.resolve())
print("processed ", OUT_DIR.resolve())

## 1. Source frames

Anything in `data/raw/` is used as it is. To fill it from a recording, the repo already
ships ffmpeg for the desktop client:

```
../../src/client/native/win32-x64/ffmpeg -i capture.mp4 -vf fps=1 data/raw/frame_%04d.png
```

One frame per second, because consecutive frames of a screen recording are nearly
identical and a hundred of them teach the model no more than one of them does.

In [ ]:
"""Draw a synthetic desktop frame: flat chrome, text-like rows, one noisy region."""

def synthesise_frame(size, rng):
    width, height = size
    background = tuple(int(v) for v in rng.integers(20, 60, size=3))
    frame = Image.new("RGB", size, background)
    draw = ImageDraw.Draw(frame)

    # A soft vertical gradient, the kind a desktop wallpaper has.
    for y in range(height):
        shade = int(18 * y / height)
        draw.line([(0, y), (width, y)], fill=tuple(min(255, c + shade) for c in background))

    for _ in range(int(rng.integers(2, 5))):
        w = int(rng.integers(width // 4, width // 2))
        h = int(rng.integers(height // 4, height // 2))
        x = int(rng.integers(0, max(1, width - w)))
        y = int(rng.integers(0, max(1, height - h)))
        panel = tuple(int(v) for v in rng.integers(180, 250, size=3))
        accent = tuple(int(v) for v in rng.integers(40, 140, size=3))

        draw.rectangle([x, y, x + w, y + h], fill=panel, outline=accent, width=2)
        draw.rectangle([x, y, x + w, y + 22], fill=accent)          # title bar
        for i in range(3):                                          # window buttons
            draw.ellipse([x + w - 18 - i * 16, y + 6, x + w - 8 - i * 16, y + 16], fill=panel)

        # Text-like rows: the high frequency detail upscaling is actually judged on.
        ink = tuple(int(v) for v in rng.integers(0, 90, size=3))
        row = y + 34
        while row < y + h - 12:
            length = int(rng.integers(w // 4, max(w // 4 + 1, w - 24)))
            draw.rectangle([x + 12, row, x + 12 + length, row + 4], fill=ink)
            row += int(rng.integers(10, 18))

    # One block of noise stands in for photo or video content playing on the screen.
    nw, nh = width // 5, height // 5
    nx, ny = int(rng.integers(0, width - nw)), int(rng.integers(0, height - nh))
    frame.paste(Image.fromarray(rng.integers(0, 255, (nh, nw, 3), dtype=np.uint8)), (nx, ny))

    return frame


sources = sorted(p for p in RAW_DIR.rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)

if not sources and SYNTHESISE_IF_EMPTY:
    print(f"{RAW_DIR} is empty — synthesising {SYNTHETIC_FRAMES} frames")
    for i in range(SYNTHETIC_FRAMES):
        path = RAW_DIR / f"synthetic_{i:04d}.png"
        synthesise_frame(SYNTHETIC_SIZE, rng).save(path)
        sources.append(path)

if not sources:
    raise RuntimeError(f"no images in {RAW_DIR} and synthesis is off")

print(f"{len(sources)} source frames")
for path in sources[:5]:
    print("  ", path.name, Image.open(path).size)

## 2. Degradation

The HR target is the source frame cropped to a multiple of `SCALE`; the LR input is that
crop resized down by `SCALE` with a bicubic filter. Cropping first is what keeps the two
aligned — a downscale of an odd width would round, and every patch pair after it would
sit half a pixel off.

A real stream is degraded by the encoder as well as by resolution, so a model trained on
clean bicubic pairs meets blocking and ringing it has never seen. Putting a JPEG or H.264
round trip in this function is the first thing to try once the basic pipeline holds.

In [ ]:
"""Load a frame, crop it onto the scale grid, and derive its LR counterpart."""

def degrade(path, scale):
    hr = Image.open(path).convert("RGB")
    width = hr.width - hr.width % scale
    height = hr.height - hr.height % scale
    hr = hr.crop((0, 0, width, height))
    lr = hr.resize((width // scale, height // scale), Image.BICUBIC)
    return np.asarray(hr), np.asarray(lr)


hr_demo, lr_demo = degrade(sources[0], SCALE)
print("HR", hr_demo.shape, hr_demo.dtype)
print("LR", lr_demo.shape, lr_demo.dtype)

## 3. Tiling

Patches are cut on the HR grid at multiples of `SCALE`, so each one maps onto an exact LR
patch with no rounding. A patch survives only if the variance of its luma clears
`VAR_MIN`: flat background carries no information about upscaling, and on a desktop frame
it is most of the pixels.

In [ ]:
"""Cut aligned LR/HR patch pairs out of one frame, keeping the detailed ones."""

LUMA = np.array([0.299, 0.587, 0.114], dtype=np.float32)


def tile(hr, lr, rng):
    pairs = []
    for y in range(0, hr.shape[0] - PATCH_HR + 1, STRIDE_HR):
        for x in range(0, hr.shape[1] - PATCH_HR + 1, STRIDE_HR):
            y_hr, x_hr = y - y % SCALE, x - x % SCALE
            hr_patch = hr[y_hr:y_hr + PATCH_HR, x_hr:x_hr + PATCH_HR]
            if (hr_patch.astype(np.float32) @ LUMA).var() < VAR_MIN:
                continue
            y_lr, x_lr = y_hr // SCALE, x_hr // SCALE
            pairs.append((lr[y_lr:y_lr + PATCH_LR, x_lr:x_lr + PATCH_LR], hr_patch))

    if len(pairs) > MAX_PATCHES_PER_FRAME:
        keep = rng.choice(len(pairs), MAX_PATCHES_PER_FRAME, replace=False)
        pairs = [pairs[i] for i in keep]
    return pairs


print(f"{sources[0].name}: {len(tile(hr_demo, lr_demo, rng))} patches kept")

In [ ]:
"""Run every frame through degrade + tile, split by frame, and stack the patches."""

order = rng.permutation(len(sources))
n_val = max(1, round(len(sources) * VAL_FRACTION)) if len(sources) > 1 else 0
val_frames = {sources[i] for i in order[:n_val]}

buckets = {"train": {"lr": [], "hr": [], "frames": []},
           "val": {"lr": [], "hr": [], "frames": []}}

start = time.time()
for path in sources:
    split = "val" if path in val_frames else "train"
    hr, lr = degrade(path, SCALE)
    pairs = tile(hr, lr, rng)
    if not pairs:
        print(f"  skipped {path.name} — no patch cleared VAR_MIN")
        continue
    buckets[split]["lr"].extend(pair[0] for pair in pairs)
    buckets[split]["hr"].extend(pair[1] for pair in pairs)
    buckets[split]["frames"].append(path.name)

for split, bucket in buckets.items():
    bucket["lr"] = (np.stack(bucket["lr"]) if bucket["lr"]
                    else np.empty((0, PATCH_LR, PATCH_LR, 3), np.uint8))
    bucket["hr"] = (np.stack(bucket["hr"]) if bucket["hr"]
                    else np.empty((0, PATCH_HR, PATCH_HR, 3), np.uint8))
    print(f"{split:5} {len(bucket['frames']):3} frames  {len(bucket['lr']):5} patches  "
          f"lr {bucket['lr'].shape}  hr {bucket['hr'].shape}")

print(f"tiled in {time.time() - start:.1f}s")

if len(buckets["train"]["lr"]) == 0:
    raise RuntimeError("no training patches — lower VAR_MIN or add more frames")

## 4. Writing the set

Four `.npy` arrays and a manifest. `uint8` on purpose: a float32 copy is four times the
bytes for no extra information, and the training notebook divides by 255 on the way to
the device anyway.

In [ ]:
"""Save the arrays and the manifest that describes them."""

for split, bucket in buckets.items():
    np.save(OUT_DIR / f"{split}_lr.npy", bucket["lr"])
    np.save(OUT_DIR / f"{split}_hr.npy", bucket["hr"])

manifest = {
    "created": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "scale": SCALE,
    "patch_lr": PATCH_LR,
    "patch_hr": PATCH_HR,
    "stride_hr": STRIDE_HR,
    "var_min": VAR_MIN,
    "max_patches_per_frame": MAX_PATCHES_PER_FRAME,
    "seed": SEED,
    "degradation": "bicubic downscale",
    "splits": {
        split: {
            "patches": int(len(bucket["lr"])),
            "frames": sorted(bucket["frames"]),
            "lr": f"{split}_lr.npy",
            "hr": f"{split}_hr.npy",
        }
        for split, bucket in buckets.items()
    },
}

(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=4), encoding="utf-8")

written = sum(p.stat().st_size for p in OUT_DIR.glob("*.npy"))
print(f"{written / 1e6:.1f} MB written to {OUT_DIR.resolve()}")
print(json.dumps({k: v for k, v in manifest.items() if k != "splits"}, indent=4))

## 5. Read it back

The check that matters is alignment: an LR patch and its HR patch have to show the same
content. If the pipeline is off by a crop or a rounding step it shows here as a shift
between the two columns, and nothing downstream would tell you — the model would simply
train to a low ceiling.

In [ ]:
"""Reload from disk and look at the pairs."""

manifest = json.loads((OUT_DIR / "manifest.json").read_text(encoding="utf-8"))
lr = np.load(OUT_DIR / manifest["splits"]["train"]["lr"])
hr = np.load(OUT_DIR / manifest["splits"]["train"]["hr"])
print("reloaded", lr.shape, hr.shape, lr.dtype)

rows = min(4, len(lr))
picks = np.random.default_rng(SEED).choice(len(lr), rows, replace=False)

fig, axes = plt.subplots(rows, 2, figsize=(5, 2.5 * rows))
axes = np.atleast_2d(axes)
for row, index in enumerate(picks):
    axes[row][0].imshow(lr[index], interpolation="nearest")
    axes[row][0].set_title(f"LR {lr.shape[1]}px", fontsize=9)
    axes[row][1].imshow(hr[index], interpolation="nearest")
    axes[row][1].set_title(f"HR {hr.shape[1]}px", fontsize=9)
    for ax in axes[row]:
        ax.axis("off")
fig.suptitle("patch pairs — the two columns must show the same content")
fig.tight_layout()
plt.show()

In [ ]:
"""Luma variance of the kept patches, against the threshold that let them through."""

variance = (hr.astype(np.float32) @ LUMA).var(axis=(1, 2))

plt.figure(figsize=(6, 3))
plt.hist(variance, bins=40)
plt.axvline(VAR_MIN, color="red", label=f"VAR_MIN = {VAR_MIN}")
plt.xlabel("luma variance of HR patch")
plt.ylabel("patches")
plt.legend()
plt.tight_layout()
plt.show()

print(f"variance  min {variance.min():.0f}  median {np.median(variance):.0f}  max {variance.max():.0f}")
print("\nnext: upscale_dummy.ipynb")